In [3]:
import pandas as pd
import numpy as np

In [4]:
df_raw = pd.DataFrame([
    {"pedido_id": "P001", "tipo_entrega": "Moto", "tempo_entrega": 24, "valor_pedido": 45.0, "avaliacao": 5.0},

    {"pedido_id": "P002", "tipo_entrega": "Carro", "tempo_entrega": 26, "valor_pedido": np.nan, "avaliacao": 4.0},

    {"pedido_id": "P003", "tipo_entrega": "Moto", "tempo_entrega": 28, "valor_pedido": 62.0, "avaliacao": 5.0},

    {"pedido_id": "P004", "tipo_entrega": "Bike", "tempo_entrega": 30, "valor_pedido": 38.0, "avaliacao": 3.0},

    {"pedido_id": "P005", "tipo_entrega": "Carro", "tempo_entrega": 32, "valor_pedido": 80.0, "avaliacao": np.nan},

    {"pedido_id": "P006", "tipo_entrega": "Moto", "tempo_entrega": 180, "valor_pedido": 55.0, "avaliacao": 2.0},

    {"pedido_id": "P007", "tipo_entrega": "Bike", "tempo_entrega": 30, "valor_pedido": 38.0, "avaliacao": 3.0},

])

df_raw

,pedido_id,tipo_entrega,tempo_entrega,valor_pedido,avaliacao
0,P001,Moto,24,45.0,5.0
1,P002,Carro,26,NaN,4.0
2,P003,Moto,28,62.0,5.0
3,P004,Bike,30,38.0,3.0
4,P005,Carro,32,80.0,NaN
5,P006,Moto,180,55.0,2.0
6,P007,Bike,30,38.0,3.0


In [5]:
print("Total de linhas:", len(df_raw))
print("Duplicadas:", df_raw.duplicated().sum())
print("Nulos por coluna:")
print(df_raw.isna().sum())

Total de linhas: 7
Duplicadas: 0
Nulos por coluna:
pedido_id        0
tipo_entrega     0
tempo_entrega    0
valor_pedido     1
avaliacao        1
dtype: int64


In [6]:
# Remove duplicadas e preenche nulos numericos com mediana
df_clean = df_raw.drop_duplicates().copy()

for col in ["tempo_entrega", "valor_pedido", "avaliacao"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

df_clean

,pedido_id,tipo_entrega,tempo_entrega,valor_pedido,avaliacao
0,P001,Moto,24,45.0,5.0
1,P002,Carro,26,50.0,4.0
2,P003,Moto,28,62.0,5.0
3,P004,Bike,30,38.0,3.0
4,P005,Carro,32,80.0,3.5
5,P006,Moto,180,55.0,2.0
6,P007,Bike,30,38.0,3.0


In [7]:
# Tratar outlier em tempo de entrega usando quantis (Q1 e Q3)

# Qualtil é um ponto de corte da distribuição
# - quantile(0.25) => Q1 : Valor abaixo de ~25%
# - quantile(0.75) => Q3 : Valor abaixo de ~75%
q1 = df_clean["tempo_entrega"].quantile(0.25)
q3 = df_clean["tempo_entrega"].quantile(0.75)

# IQR (Interavlo Interquartil) mede a faixa central dos dados
# IQR = Q3 - Q1
iqr = q3 - q1

# Limites classicos
# Valor abaixo de Lower ou acima de Upper considerado discrepante
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df_clean["tempo_entrega_tratado"] = df_clean["tempo_entrega"].clip(lower=lower, upper=upper)

alterados = (df_clean["tempo_entrega"] != df_clean["tempo_entrega_tratado"]).sum()

print(f"Q1 (25%): {q1:.2f}")
print(f"Q3 (75%): {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Limite inferior: {lower:.2f}")
print(f"Limite superior: {upper:.2f}")
print(f"Tempos alterados pelo clipping: {alterados}")

df_clean[["pedido_id", "tempo_entrega", "tempo_entrega_tratado"]]

Q1 (25%): 27.00
Q3 (75%): 31.00
IQR: 4.00
Limite inferior: 21.00
Limite superior: 37.00
Tempos alterados pelo clipping: 1


,pedido_id,tempo_entrega,tempo_entrega_tratado
0,P001,24,24
1,P002,26,26
2,P003,28,28
3,P004,30,30
4,P005,32,32
5,P006,180,37
6,P007,30,30


In [ ]:
# Relacao entre colunas numericas

# Correlacao indica como duas variaveis mudam juntas.
# Valor proximo de +1: quando uma sobe, a outra tende a subir.
# Valor proximo de -1: quando uma sobe, a outra tende a descer.
# Valor proximo de 0: pouca relacao linear entre elas.
df_clean[["tempo_entrega_tratado", "valor_pedido", "avaliacao"]].corr()

,tempo_entrega_tratado,valor_pedido,avaliacao
tempo_entrega_tratado,1.000000,0.286611,-0.855088
valor_pedido,0.286611,1.000000,0.125375
avaliacao,-0.855088,0.125375,1.000000
